<a href="https://colab.research.google.com/github/YUKI-0615/cowc/blob/main/SAHI(%E7%99%BD%E5%B7%9D)_%E3%81%AE%E3%82%B3%E3%83%94%E3%83%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 必要なライブラリのインストール
!pip install sahi ultralytics yolov5 > /dev/null
from google.colab import drive
import os

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.20.0 requires huggingface-hub>=0.25.0, but you have huggingface-hub 0.24.7 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.24.7 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.24.7 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.24.7 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 0.24.7 which is incompatible.
Mounted at /content/drive


In [ ]:
import torch
import sys

# ==========================================
# 🚑 PyTorch 2.6 エラー回避パッチ
# ==========================================
# PyTorch 2.6以降、weights_only=Trueがデフォルトになり、
# YOLOv5のようなカスタムモデル構造を含むファイルの読み込みがブロックされるため、
# 一時的にデフォルトを weights_only=False (制限なし) に書き換えます。

_original_torch_load = torch.load

def patched_torch_load(*args, **kwargs):
    # weights_only が指定されていない場合、強制的に False にする
    if 'weights_only' not in kwargs:
        kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)

# パッチを適用
torch.load = patched_torch_load
print("✅ PyTorch load patch applied (weights_only=False allowed)")

# ==========================================
# 📂 パス設定（念のため再定義）
# ==========================================
# SAHIが内部でYOLOv5のコードを参照するため、パスを通しておきます
YOLO5_PATH = '/content/drive/MyDrive/yolov5_cowc/yolov5'
if YOLO5_PATH not in sys.path:
    sys.path.append(YOLO5_PATH)

# ==========================================
# 🚀 これでモデルロードを実行してください
# ==========================================
# (ここにSAHIのロードコードを続けて書くか、次のセルで実行してください)

✅ PyTorch load patch applied (weights_only=False allowed)


In [ ]:
# 1. データをDriveからローカルにコピーして解凍
print("データをローカルにコピー中...")

# コピーしたいファイル名を指定
ZIP_FILE_NAME = "/content/drive/MyDrive/ZIP/crop_label_original.zip" # ファイル名のみを指定
# Colab内の一時的な保存パス
LOCAL_ZIP_PATH = f"/content/drive/MyDrive/ZIP/crop_label_original.zip" # ZIP_FILE_NAMEのファイル名部分のみを使用
# 解凍先のディレクトリ
EXTRACT_PATH = "/content/datasets"

# DriveからColabにファイルをコピー
# ZIP_FILE_NAME の前に /content/drive/MyDrive/ をつける
!cp "{ZIP_FILE_NAME}" {LOCAL_ZIP_PATH}

# コピーしたファイル（{LOCAL_ZIP_PATH}）を解凍
# ( -q は解凍ログを非表示にするオプション、-d は解凍先を指定するオプション)
!unzip -q {LOCAL_ZIP_PATH} -d {EXTRACT_PATH}

print(f"コピー・解凍完了。 -> {EXTRACT_PATH} に展開されました。")

データをローカルにコピー中...
cp: '/content/drive/MyDrive/ZIP/crop_label_original.zip' and '/content/drive/MyDrive/ZIP/crop_label_original.zip' are the same file
コピー・解凍完了。 -> /content/datasets に展開されました。


In [ ]:
import os
import gc
import logging
import warnings
import sys
import contextlib
import torch
import numpy as np
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from PIL import Image
import pandas as pd

# ==========================================
# 🤫 ログ・警告の非表示設定
# ==========================================
warnings.filterwarnings("ignore")
logging.getLogger("sahi").setLevel(logging.CRITICAL)
logging.getLogger("ultralytics").setLevel(logging.CRITICAL)

# 標準出力を一時的に遮断するためのコンテキストマネージャ
@contextlib.contextmanager
def suppress_stdout_stderr():
    with open(os.devnull, 'w') as devnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = devnull
        sys.stderr = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

# ==========================================
# 🛠️ 設定
# ==========================================
VAL_IMG_DIR = '/content/datasets/crop_label_original/val/images'
VAL_LBL_DIR = '/content/datasets/crop_label_original/val/labels'
IOU_THRESH = 0.5
CONF_THRESH = 0.25
NMS_IOU_THRESH = 0.5
device_gpu = 'cuda:0' if torch.cuda.is_available() else 'cpu'

YOLOV5_WEIGHTS = "/content/drive/MyDrive/yolov5/runs/cowc/o_all/60_5/re/0.2/cowc_all/weights/best.pt"
YOLOV8_WEIGHTS = "/content/drive/MyDrive/yolov8/runs/cowc/o_all/60_5/train-2/weights/best.pt"

# ==========================================
# 📦 モデルのロード & 設定（出力を非表示）
# ==========================================
with suppress_stdout_stderr():
    detection_model_v5 = AutoDetectionModel.from_pretrained(
        model_type="yolov5",
        model_path=YOLOV5_WEIGHTS,
        confidence_threshold=CONF_THRESH,
        device=device_gpu
    )

    detection_model_v8 = AutoDetectionModel.from_pretrained(
        model_type="yolov8",
        model_path=YOLOV8_WEIGHTS,
        confidence_threshold=CONF_THRESH,
        device=device_gpu
    )

    detection_model_v5.model.iou = NMS_IOU_THRESH
    detection_model_v8.model.overrides['iou'] = NMS_IOU_THRESH

# ==========================================
# 🧮 計算用関数群
# ==========================================
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    if union == 0: return 0
    return intersection / union

def calculate_ap(rec, prec):
    mrec = np.concatenate(([0.0], rec, [1.0]))
    mpre = np.concatenate(([0.0], prec, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
    return ap

def evaluate_model(model, model_name, use_sahi):
    img_files = sorted(
        [f for f in os.listdir(VAL_IMG_DIR) if f.endswith('.png')]
    )
    all_gt = []
    all_pred = []

    for i, img_file in enumerate(img_files):
        img_path = os.path.join(VAL_IMG_DIR, img_file)
        lbl_path = os.path.join(VAL_LBL_DIR, img_file.replace('.png', '.txt'))

        with Image.open(img_path) as img:
            img_w, img_h = img.size

        # 1. GT取得 (Car=0)
        gt_boxes = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = list(map(float, line.strip().split()))
                    if int(parts[0]) == 0:
                        cx, cy, w, h = parts[1], parts[2], parts[3], parts[4]
                        gt_boxes.append([
                            (cx - w/2) * img_w,
                            (cy - h/2) * img_h,
                            (cx + w/2) * img_w,
                            (cy + h/2) * img_h
                        ])
        all_gt.append(gt_boxes)

        # 2. 推論実行（出力を完全に抑制）
        overlap_ratio = 0.2 if use_sahi else 0.0
        with torch.no_grad():
            with suppress_stdout_stderr():
                result = get_sliced_prediction(
                    img_path,
                    model,
                    slice_height=512,
                    slice_width=512,
                    overlap_height_ratio=overlap_ratio,
                    overlap_width_ratio=overlap_ratio,
                    postprocess_type="GREEDYNMM",
                    postprocess_match_metric="IOU",
                    postprocess_match_threshold=0.5,
                    postprocess_class_agnostic=False,
                    verbose=0
                )

        # 3. 予測データ取得 (car のみ)
        for obj in result.object_prediction_list:
            if obj.category.name == "car":
                bbox = obj.bbox
                score = obj.score.value
                all_pred.append([i, score, bbox.minx, bbox.miny, bbox.maxx, bbox.maxy])

        del result
        torch.cuda.empty_cache()
        gc.collect()

    # --- AP計算 ---
    if not all_pred:
        return 0.0, 0.0, 0.0

    all_pred = np.array(all_pred)
    sorted_idx = np.argsort(-all_pred[:, 1])
    all_pred = all_pred[sorted_idx]

    nd = len(all_pred)
    tp = np.zeros(nd)
    fp = np.zeros(nd)
    gt_matched = {idx: [False]*len(gts) for idx, gts in enumerate(all_gt)}

    total_gt = sum([len(gts) for gts in all_gt])
    if total_gt == 0:
        return 0.0, 0.0, 0.0

    for p in range(nd):
        img_idx = int(all_pred[p, 0])
        pred_box = all_pred[p, 2:].tolist()
        gts = all_gt[img_idx]

        best_iou = 0
        best_gt_idx = -1
        for g, gt_box in enumerate(gts):
            iou = compute_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = g

        if best_iou >= IOU_THRESH:
            if not gt_matched[img_idx][best_gt_idx]:
                tp[p] = 1
                gt_matched[img_idx][best_gt_idx] = True
            else:
                fp[p] = 1
        else:
            fp[p] = 1

    tp_cumsum = np.cumsum(tp)
    fp_cumsum = np.cumsum(fp)
    recalls = tp_cumsum / total_gt
    precisions = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, 1e-16)
    ap = calculate_ap(recalls, precisions)

    final_prec = precisions[-1] if len(precisions) > 0 else 0
    final_rec = recalls[-1] if len(recalls) > 0 else 0

    return ap, final_prec, final_rec

# ==========================================
# 🚀 実行 & 結果表示
# ==========================================
results = []
configs = [
    (detection_model_v5, "YOLOv5", False),
    (detection_model_v5, "YOLOv5", True),
    (detection_model_v8, "YOLOv8", False),
    (detection_model_v8, "YOLOv8", True),
]

for model, name, use_sahi in configs:
    ap, p, r = evaluate_model(model, name, use_sahi=use_sahi)
    mode_label = "SAHI" if use_sahi else "Normal"
    results.append([name, mode_label, ap, p, r])
    torch.cuda.empty_cache()
    gc.collect()

df = pd.DataFrame(results, columns=["Model", "Mode", "mAP@50", "Precision", "Recall"])
print(df)

KeyboardInterrupt: 

In [ ]:
import os
import gc
import logging
import warnings
import sys
import contextlib
import torch
import numpy as np
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from PIL import Image
import pandas as pd

# ==========================================
# 🤫 ログ・警告の非表示設定
# ==========================================
warnings.filterwarnings("ignore")
logging.getLogger("sahi").setLevel(logging.CRITICAL)
logging.getLogger("ultralytics").setLevel(logging.CRITICAL)

# 不要な標準出力を一時的に遮断するためのコンテキストマネージャ
@contextlib.contextmanager
def suppress_stdout_stderr():
    with open(os.devnull, 'w') as devnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = devnull
        sys.stderr = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

# ==========================================
# 🛠 設定
# ==========================================
VAL_IMG_DIR = '/content/datasets/crop_label_original/val/images'
VAL_LBL_DIR = '/content/datasets/crop_label_original/val/labels'
IOU_THRESH = 0.5
CONF_THRESH = 0.25
NMS_IOU_THRESH = 0.5
device_gpu = 'cuda:0' if torch.cuda.is_available() else 'cpu'

YOLOV5_WEIGHTS = "/content/drive/MyDrive/yolov5/runs/cowc/o_all/60_5/re/0.2/cowc_all/weights/best.pt"
YOLOV8_WEIGHTS = "/content/drive/MyDrive/yolov8/runs/cowc/o_all/60_5/train-2/weights/best.pt"

# ==========================================
# 📦 モデルのロード & 設定
# ==========================================
print("⏳ YOLOv5 モデルをロード中...")
with suppress_stdout_stderr():
    detection_model_v5 = AutoDetectionModel.from_pretrained(
        model_type="yolov5",
        model_path=YOLOV5_WEIGHTS,
        confidence_threshold=CONF_THRESH,
        device=device_gpu
    )
    detection_model_v5.model.iou = NMS_IOU_THRESH

print("⏳ YOLOv8 モデルをロード中...")
with suppress_stdout_stderr():
    detection_model_v8 = AutoDetectionModel.from_pretrained(
        model_type="yolov8",
        model_path=YOLOV8_WEIGHTS,
        confidence_threshold=CONF_THRESH,
        device=device_gpu
    )
    detection_model_v8.model.overrides['iou'] = NMS_IOU_THRESH

print("✅ モデルのロードが完了しました。\n")

# ==========================================
# 🧮 計算用関数群
# ==========================================
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    if union == 0: return 0
    return intersection / union

def calculate_ap(rec, prec):
    mrec = np.concatenate(([0.0], rec, [1.0]))
    mpre = np.concatenate(([0.0], prec, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
    return ap

def evaluate_model(model, model_name, use_sahi):
    img_files = sorted(
        [f for f in os.listdir(VAL_IMG_DIR) if f.endswith('.png')]
    )
    all_gt = []
    all_pred = []

    for i, img_file in enumerate(img_files):
        img_path = os.path.join(VAL_IMG_DIR, img_file)
        lbl_path = os.path.join(VAL_LBL_DIR, img_file.replace('.png', '.txt'))

        with Image.open(img_path) as img:
            img_w, img_h = img.size

        # 1. GT取得 (Car=0)
        gt_boxes = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = list(map(float, line.strip().split()))
                    if int(parts[0]) == 0:
                        cx, cy, w, h = parts[1], parts[2], parts[3], parts[4]
                        gt_boxes.append([
                            (cx - w/2) * img_w,
                            (cy - h/2) * img_h,
                            (cx + w/2) * img_w,
                            (cy + h/2) * img_h
                        ])
        all_gt.append(gt_boxes)

        # 2. 推論実行
        overlap_ratio = 0.2 if use_sahi else 0.0
        with torch.no_grad():
            with suppress_stdout_stderr():
                result = get_sliced_prediction(
                    img_path,
                    model,
                    slice_height=512,
                    slice_width=512,
                    overlap_height_ratio=overlap_ratio,
                    overlap_width_ratio=overlap_ratio,
                    postprocess_type="GREEDYNMM",
                    postprocess_match_metric="IOU",
                    postprocess_match_threshold=0.5,
                    postprocess_class_agnostic=False,
                    verbose=0
                )

        # 3. 予測データ取得 (car のみ)
        for obj in result.object_prediction_list:
            if obj.category.name == "car":
                bbox = obj.bbox
                score = obj.score.value
                all_pred.append([i, score, bbox.minx, bbox.miny, bbox.maxx, bbox.maxy])

        del result
        torch.cuda.empty_cache()
        gc.collect()

    # --- AP計算 ---
    if not all_pred:
        return 0.0, 0.0, 0.0

    all_pred = np.array(all_pred)
    sorted_idx = np.argsort(-all_pred[:, 1])
    all_pred = all_pred[sorted_idx]

    nd = len(all_pred)
    tp = np.zeros(nd)
    fp = np.zeros(nd)
    gt_matched = {idx: [False]*len(gts) for idx, gts in enumerate(all_gt)}

    total_gt = sum([len(gts) for gts in all_gt])
    if total_gt == 0:
        return 0.0, 0.0, 0.0

    for p in range(nd):
        img_idx = int(all_pred[p, 0])
        pred_box = all_pred[p, 2:].tolist()
        gts = all_gt[img_idx]

        best_iou = 0
        best_gt_idx = -1
        for g, gt_box in enumerate(gts):
            iou = compute_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = g

        if best_iou >= IOU_THRESH:
            if not gt_matched[img_idx][best_gt_idx]:
                tp[p] = 1
                gt_matched[img_idx][best_gt_idx] = True
            else:
                fp[p] = 1
        else:
            fp[p] = 1

    tp_cumsum = np.cumsum(tp)
    fp_cumsum = np.cumsum(fp)
    recalls = tp_cumsum / total_gt
    precisions = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, 1e-16)
    ap = calculate_ap(recalls, precisions)

    final_prec = precisions[-1] if len(precisions) > 0 else 0
    final_rec = recalls[-1] if len(recalls) > 0 else 0

    return ap, final_prec, final_rec

# ==========================================
# 🚀 実行 & 結果表示
# ==========================================
results = []
configs = [
    (detection_model_v5, "YOLOv5", False),
    (detection_model_v5, "YOLOv5", True),
    (detection_model_v8, "YOLOv8", False),
    (detection_model_v8, "YOLOv8", True),
]

for model, name, use_sahi in configs:
    mode_label = "SAHI" if use_sahi else "Normal"
    print(f"🔄 実行中: {name} [{mode_label} モード] の評価を行っています...")

    ap, p, r = evaluate_model(model, name, use_sahi=use_sahi)
    results.append([name, mode_label, ap, p, r])

    # 完了後にその場の精度を表示
    print(f"  └ 完了! mAP@50: {ap:.4f} | Precision: {p:.4f} | Recall: {r:.4f}\n")

    torch.cuda.empty_cache()
    gc.collect()

print("="*50)
print("📊 最終集計結果:")
print("="*50)
df = pd.DataFrame(results, columns=["Model", "Mode", "mAP@50", "Precision", "Recall"])
print(df)

⏳ YOLOv5 モデルをロード中...
⏳ YOLOv8 モデルをロード中...
✅ モデルのロードが完了しました。

🔄 実行中: YOLOv5 [Normal モード] の評価を行っています...
  └ 完了! mAP@50: 0.8773 | Precision: 0.8690 | Recall: 0.9020

🔄 実行中: YOLOv5 [SAHI モード] の評価を行っています...
  └ 完了! mAP@50: 0.9056 | Precision: 0.8655 | Recall: 0.9274

🔄 実行中: YOLOv8 [Normal モード] の評価を行っています...
  └ 完了! mAP@50: 0.8443 | Precision: 0.8978 | Recall: 0.8673

🔄 実行中: YOLOv8 [SAHI モード] の評価を行っています...
  └ 完了! mAP@50: 0.8749 | Precision: 0.8958 | Recall: 0.8970

📊 最終集計結果:
    Model    Mode    mAP@50  Precision    Recall
0  YOLOv5  Normal  0.877269   0.868995  0.901987
1  YOLOv5    SAHI  0.905646   0.865507  0.927406
2  YOLOv8  Normal  0.844338   0.897809  0.867252
3  YOLOv8    SAHI  0.874852   0.895773  0.896979


In [ ]:
import os
import gc
import logging
import warnings
import sys
import contextlib
import torch
import numpy as np
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from PIL import Image
import pandas as pd

# ==========================================
# 🤫 ログ・警告の非表示設定
# ==========================================
warnings.filterwarnings("ignore")
logging.getLogger("sahi").setLevel(logging.CRITICAL)
logging.getLogger("ultralytics").setLevel(logging.CRITICAL)

# 不要な標準出力を一時的に遮断するためのコンテキストマネージャ
@contextlib.contextmanager
def suppress_stdout_stderr():
    with open(os.devnull, 'w') as devnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = devnull
        sys.stderr = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

# ==========================================
# 🛠 設定
# ==========================================
VAL_IMG_DIR = '/content/datasets/crop_label_original/val/images'
VAL_LBL_DIR = '/content/datasets/crop_label_original/val/labels'
IOU_THRESH = 0.5
CONF_THRESH = 0.25
NMS_IOU_THRESH = 0.5
device_gpu = 'cuda:0' if torch.cuda.is_available() else 'cpu'

YOLOV5_WEIGHTS = "/content/drive/MyDrive/yolov5/runs/cowc/o_all/60_5/re/0.2/cowc_all/weights/best.pt"
YOLO26_WEIGHTS = "/content/drive/MyDrive/yolo26/runs/cowc/o/30_5/weights/best.pt"

# ==========================================
# 📦 モデルのロード & 設定
# ==========================================
print("⏳ YOLOv5 モデルをロード中...")
with suppress_stdout_stderr():
    detection_model_v5 = AutoDetectionModel.from_pretrained(
        model_type="yolov5",
        model_path=YOLOV5_WEIGHTS,
        confidence_threshold=CONF_THRESH,
        device=device_gpu
    )
    detection_model_v5.model.iou = NMS_IOU_THRESH

print("⏳ YOLO26 モデルをロード中...")
with suppress_stdout_stderr():
    detection_model_yolo26 = AutoDetectionModel.from_pretrained(
        model_type="yolo26",
        model_path=YOLO26_WEIGHTS,
        confidence_threshold=CONF_THRESH,
        device=device_gpu
    )
    detection_model_yolo26.model.overrides['iou'] = NMS_IOU_THRESH

print("✅ モデルのロードが完了しました。\n")

# ==========================================
# 🧮 計算用関数群
# ==========================================
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection

    if union == 0: return 0
    return intersection / union

def calculate_ap(rec, prec):
    mrec = np.concatenate(([0.0], rec, [1.0]))
    mpre = np.concatenate(([0.0], prec, [0.0]))
    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])
    i = np.where(mrec[1:] != mrec[:-1])[0]
    ap = np.sum((mrec[i + 1] - mrec[i]) * mpre[i + 1])
    return ap

def evaluate_model(model, model_name, use_sahi):
    img_files = sorted(
        [f for f in os.listdir(VAL_IMG_DIR) if f.endswith('.png')]
    )
    all_gt = []
    all_pred = []

    for i, img_file in enumerate(img_files):
        img_path = os.path.join(VAL_IMG_DIR, img_file)
        lbl_path = os.path.join(VAL_LBL_DIR, img_file.replace('.png', '.txt'))

        with Image.open(img_path) as img:
            img_w, img_h = img.size

        # 1. GT取得 (Car=0)
        gt_boxes = []
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f:
                    parts = list(map(float, line.strip().split()))
                    if int(parts[0]) == 0:
                        cx, cy, w, h = parts[1], parts[2], parts[3], parts[4]
                        gt_boxes.append([
                            (cx - w/2) * img_w,
                            (cy - h/2) * img_h,
                            (cx + w/2) * img_w,
                            (cy + h/2) * img_h
                        ])
        all_gt.append(gt_boxes)

        # 2. 推論実行
        overlap_ratio = 0.2 if use_sahi else 0.0
        with torch.no_grad():
            with suppress_stdout_stderr():
                result = get_sliced_prediction(
                    img_path,
                    model,
                    slice_height=512,
                    slice_width=512,
                    overlap_height_ratio=overlap_ratio,
                    overlap_width_ratio=overlap_ratio,
                    postprocess_type="GREEDYNMM",
                    postprocess_match_metric="IOU",
                    postprocess_match_threshold=0.5,
                    postprocess_class_agnostic=False,
                    verbose=0
                )

        # 3. 予測データ取得 (car のみ)
        for obj in result.object_prediction_list:
            if obj.category.name == "car":
                bbox = obj.bbox
                score = obj.score.value
                all_pred.append([i, score, bbox.minx, bbox.miny, bbox.maxx, bbox.maxy])

        del result
        torch.cuda.empty_cache()
        gc.collect()

    # --- AP計算 ---
    if not all_pred:
        return 0.0, 0.0, 0.0

    all_pred = np.array(all_pred)
    sorted_idx = np.argsort(-all_pred[:, 1])
    all_pred = all_pred[sorted_idx]

    nd = len(all_pred)
    tp = np.zeros(nd)
    fp = np.zeros(nd)
    gt_matched = {idx: [False]*len(gts) for idx, gts in enumerate(all_gt)}

    total_gt = sum([len(gts) for gts in all_gt])
    if total_gt == 0:
        return 0.0, 0.0, 0.0

    for p in range(nd):
        img_idx = int(all_pred[p, 0])
        pred_box = all_pred[p, 2:].tolist()
        gts = all_gt[img_idx]

        best_iou = 0
        best_gt_idx = -1
        for g, gt_box in enumerate(gts):
            iou = compute_iou(pred_box, gt_box)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = g

        if best_iou >= IOU_THRESH:
            if not gt_matched[img_idx][best_gt_idx]:
                tp[p] = 1
                gt_matched[img_idx][best_gt_idx] = True
            else:
                fp[p] = 1
        else:
            fp[p] = 1

    tp_cumsum = np.cumsum(tp)
    fp_cumsum = np.cumsum(fp)
    recalls = tp_cumsum / total_gt
    precisions = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, 1e-16)
    ap = calculate_ap(recalls, precisions)

    final_prec = precisions[-1] if len(precisions) > 0 else 0
    final_rec = recalls[-1] if len(recalls) > 0 else 0

    return ap, final_prec, final_rec

# ==========================================
# 🚀 実行 & 結果表示
# ==========================================
results = []
configs = [
    (detection_model_v5, "YOLOv5", False),
    (detection_model_v5, "YOLOv5", True),
    (detection_model_yolo26, "YOLO26", False),
    (detection_model_yolo26, "YOLO26", True),
]

for model, name, use_sahi in configs:
    mode_label = "SAHI" if use_sahi else "Normal"
    print(f"🔄 実行中: {name} [{mode_label} モード] の評価を行っています...")

    ap, p, r = evaluate_model(model, name, use_sahi=use_sahi)
    results.append([name, mode_label, ap, p, r])

    print(f"  └ 完了! mAP@50: {ap:.4f} | Precision: {p:.4f} | Recall: {r:.4f}\n")

    torch.cuda.empty_cache()
    gc.collect()

print("="*50)
print("📊 最終集計結果:")
print("="*50)
df = pd.DataFrame(results, columns=["Model", "Mode", "mAP@50", "Precision", "Recall"])
print(df)